# Test: balkberäkning (balk)

Kontrollerar `kb.balk` mot handboksformler för statiskt bestämda och obestämda balkar. Laster är positiva nedåt, M positivt när underkanten är dragen och w positiv nedåt.

Kör med **Run All**. Kerneln ska vara `.venv` i repot.

- ✔ kontrollen gick igenom
- ✘ känt fel: fel som redan är kända och inte rättade än
- ✘ FEL: nytt fel

In [ ]:
import math
import kthbygg as kb
from kontroll import kolla, kolla_fel, klass, utskrift, sammanfattning, nollställ

nollställ()

## Indata

L = 6, q = 10, P = 20, EI = 1000 (godtyckliga men konsekventa enheter).

In [ ]:
from kthbygg.balk import Balk
L, q, P, EI = 6.0, 10.0, 20.0, 1000.0

## Fritt upplagd balk, jämnt utbredd last

R = qL/2, M_max = qL²/8 i mitten, w_max = 5qL⁴/(384EI).

In [ ]:
r = Balk(L, EI).stöd(0, "led").stöd(L, "rull").utbredd_last(q).lös()
kolla("R_A", lambda: r.reaktioner[0]["V"], q * L / 2)
kolla("M_max", lambda: r.extremvärden("M")[0][1], q * L**2 / 8)
kolla("x för M_max", lambda: r.extremvärden("M")[0][0], L / 2)
kolla("w_max", lambda: r.extremvärden("w")[0][1], 5 * q * L**4 / (384 * EI))
kolla("statiskt bestämd", lambda: r.grad, 0)

## Fritt upplagd balk, punktlast i a = 2

M = Pab/L och w = Pa²b²/(3EIL) under lasten. Tvärkraften gör ett språng på P.

In [ ]:
a, b = 2.0, 4.0
r = Balk(L, EI).stöd(0, "led").stöd(L, "rull").punktlast(a, P).lös()
kolla("M(a)", lambda: r.M(a), P * a * b / L)
kolla("w(a)", lambda: r.w(a), P * a**2 * b**2 / (3 * EI * L))
kolla("V strax vänster om lasten", lambda: r.V(a, "vänster"), P * b / L)
kolla("V strax höger om lasten", lambda: r.V(a), -P * a / L)

## Konsol med punktlast i spetsen

Inspänningsmoment PL (moturs), M(0) = −PL, w(L) = PL³/(3EI).

In [ ]:
r = Balk(L, EI).stöd(0, "fast").punktlast(L, P).lös()
kolla("R_V", lambda: r.reaktioner[0]["V"], P)
kolla("R_M", lambda: r.reaktioner[0]["M"], P * L)
kolla("M(0)", lambda: r.M(0), -P * L)
kolla("w(L)", lambda: r.w(L), P * L**3 / (3 * EI))

## Fast inspänd i båda ändar, jämn last

M_ände = −qL²/12, M_mitt = qL²/24, w_mitt = qL⁴/(384EI). Två gånger statiskt obestämd.

In [ ]:
r = Balk(L, EI).stöd(0, "fast").stöd(L, "fast").utbredd_last(q).lös()
kolla("M(0)", lambda: r.M(0), -q * L**2 / 12)
kolla("M(L/2)", lambda: r.M(L / 2), q * L**2 / 24)
kolla("w(L/2)", lambda: r.w(L / 2), q * L**4 / (384 * EI))
kolla("grad 2", lambda: r.grad, 2)

## Inspänd och rullager, jämn last

R_B = 3qL/8, M(0) = −qL²/8, M_max = 9qL²/128 vid x = 5L/8. En gång statiskt obestämd.

In [ ]:
r = Balk(L, EI).stöd(0, "fast").stöd(L, "rull").utbredd_last(q).lös()
kolla("R_B", lambda: r.reaktioner[L]["V"], 3 * q * L / 8)
kolla("M(0)", lambda: r.M(0), -q * L**2 / 8)
kolla("M_max", lambda: r.extremvärden("M")[0][1], 9 * q * L**2 / 128)
kolla("x för M_max", lambda: r.extremvärden("M")[0][0], 5 * L / 8)
kolla("grad 1", lambda: r.grad, 1)

## Kontinuerlig balk över två lika fack

Mittstöd 10qL/8, ändstöd 3qL/8, stödmoment −qL²/8. Största nedböjning 0,00542·qL⁴/EI vid x = 0,4215L.

In [ ]:
r = Balk(2 * L, EI).stöd(0, "led").stöd(L, "rull").stöd(2 * L, "rull").utbredd_last(q).lös()
kolla("R mittstöd", lambda: r.reaktioner[L]["V"], 10 * q * L / 8)
kolla("R ändstöd", lambda: r.reaktioner[0]["V"], 3 * q * L / 8)
kolla("M över mittstöd", lambda: r.M(L), -q * L**2 / 8)
kolla("w_max", lambda: r.extremvärden("w")[0][1], 0.00542 * q * L**4 / EI, rel=2e-3)
kolla("x för w_max", lambda: r.extremvärden("w")[0][0], 0.4215 * L, rel=2e-3)

## Gerberbalk med inre led

Inspänd i 0, led i 4, rullager i 6, q = 10 över hela balken. Delen 4–6 är fritt upplagd och ger 10 i leden. Konsolen 0–4 bär q och 10 i spetsen: R = 50, M_inspänning = 10·4²/2 + 10·4 = 120.

In [ ]:
r = Balk(6, EI).stöd(0, "fast").led(4).stöd(6, "rull").utbredd_last(q).lös()
kolla("R_V(0)", lambda: r.reaktioner[0]["V"], 50)
kolla("R_M(0)", lambda: r.reaktioner[0]["M"], 120)
kolla("M(0)", lambda: r.M(0), -120)
kolla("M = 0 i leden", lambda: abs(r.M(4)) < 1e-9, True)
kolla("R(6)", lambda: r.reaktioner[6]["V"], 10)
kolla("statiskt bestämd", lambda: r.grad, 0)

## Punktmoment mitt på fritt upplagd balk

M0 = 12 moturs ger R_A = M0/L uppåt. Momentet gör ett språng från M0/2 till −M0/2.

In [ ]:
M0 = 12.0
r = Balk(L, EI).stöd(0, "led").stöd(L, "rull").moment(L / 2, M0).lös()
kolla("R_A", lambda: r.reaktioner[0]["V"], M0 / L)
kolla("M strax vänster", lambda: r.M(L / 2, "vänster"), M0 / 2)
kolla("M strax höger", lambda: r.M(L / 2), -M0 / 2)

## Triangellast från 0 till q

R_A = qL/6, R_B = qL/3, M_max = qL²/(9√3) vid x = L/√3.

In [ ]:
r = Balk(L, EI).stöd(0, "led").stöd(L, "rull").utbredd_last(0, q2=q).lös()
kolla("R_A", lambda: r.reaktioner[0]["V"], q * L / 6)
kolla("R_B", lambda: r.reaktioner[L]["V"], q * L / 3)
kolla("M_max", lambda: r.extremvärden("M")[0][1], q * L**2 / (9 * math.sqrt(3)))
kolla("x för M_max", lambda: r.extremvärden("M")[0][0], L / math.sqrt(3))

## Jämvikt för en blandad balk

Summan av vertikala reaktioner ska vara lika med summan av lasterna.

In [ ]:
b = Balk(10, 20e3).stöd(0, "fast").led(4).stöd(7, "rull").stöd(10, "rull")
b.utbredd_last(8, 0, 7).punktlast(8.5, 30).utbredd_last(0, 7, 10, q2=12)
r = b.lös()
kolla("ΣV = 8·7 + 30 + 12·3/2", lambda: sum(R["V"] for R in r.reaktioner.values()), 104)
kolla("M = 0 i leden", lambda: abs(r.M(4)) < 1e-9, True)

## Normalkraft och felhantering

In [ ]:
r = Balk(L, EI, EA=1e5).stöd(0, "led").stöd(L, "rull").punktlast(L, 0, H=15).lös()
kolla("N av dragande last", lambda: r.N(3), 15)
kolla("R_H", lambda: r.reaktioner[0]["H"], -15)
kolla_fel("mekanism: bara ett rullager", lambda: Balk(L).stöd(3, "rull").punktlast(1, 1).lös())
kolla_fel("horisontell last utan fixlager", lambda: Balk(L).stöd(0, "rull").stöd(L, "rull").punktlast(2, 0, H=1).lös())
kolla_fel("led i inspänning", lambda: Balk(L).stöd(0, "fast").stöd(3, "fast").led(3).lös())
kolla_fel("last utanför balken", lambda: Balk(L).punktlast(7, 1))
kolla_fel("okänd stödtyp", lambda: Balk(L).stöd(0, "gungbräda"))

## Diagram

`rita()` ritar balken och valda diagram, standard är N, V, M och w. `rita_interaktiv()` ger en plotly-figur med samma diagram där muspekaren visar alla värden i samma x. Kontrollerna räknar diagrammen; figurerna visas längst ner.

In [ ]:
import matplotlib.pyplot as plt
r = Balk(6, EI).stöd(0, "fast").led(4).stöd(6, "rull").utbredd_last(q).lös()
kolla("rita(): balk + 4 diagram", lambda: len(r.rita().axes), 5)
kolla("rita('VM', visa_balk=False)", lambda: len(r.rita("VM", visa_balk=False).axes), 2)
kolla("rita_M(): balk + M", lambda: len(r.rita_M().axes), 2)
kolla_fel("rita med okänt diagram", lambda: r.rita("X"))
plt.close("all")
r.rita(dragen_sida=True)

try:
    import plotly
except ImportError:
    plotly = None
if plotly is None:
    print("plotly är inte installerat, hoppar över rita_interaktiv")
else:
    fig = r.rita_interaktiv()
    kolla("rita_interaktiv: en kurva per diagram", lambda: sum(t.mode == "lines" for t in fig.data), 4)
    kolla("rita_interaktiv: hover över alla diagram",
          lambda: (fig.layout.hovermode, fig.layout.hoversubplots), ("x unified", "axis"))
    fig.show()

## Sammanfattning

Cellen stoppar med `AssertionError` om någon kontroll som inte är märkt som känt fel misslyckas.

In [ ]:
sammanfattning()